# 🎭 Poetry RNN Language Model — GPU-Accelerated Colab

**Architecture:** Char-level RNNLM — Embedding → 2-layer GRU → Bahdanau Attention → Softmax  
**Backend:** CuPy on GPU (falls back to NumPy automatically if no GPU found)  
**Data layout expected in Drive:**
```
MyDrive/
  data/
    raw/
      topics/
        <genre_1>/  *.txt
        <genre_2>/  *.txt
        ...
```
> **Runtime → Change runtime type → T4 GPU** before running.

## 1 · Setup

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import cupy as cp
    xp = cp
    print(f'CuPy {cp.__version__} — GPU: {cp.cuda.runtime.getDeviceProperties(0)["name"].decode()}')
except Exception:
    print('CuPy not available — falling back to NumPy (CPU). Switch to a GPU runtime for speed.')
    import numpy as cp
    xp = cp

import numpy as np
import os, re, random, time, json, math
from pathlib import Path
import matplotlib.pyplot as plt

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = Path('/content/drive/MyDrive')
DATA_DIR    = DRIVE_ROOT / 'data'
GENRES_DIR  = DATA_DIR / 'raw' / 'topics'
CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

if not GENRES_DIR.exists():
    raise FileNotFoundError(
        f'Expected data at {GENRES_DIR}. '
        'Make sure your Drive has data/raw/topics/<genre>/*.txt'
    )
genres = [d.name for d in sorted(GENRES_DIR.iterdir()) if d.is_dir()]
print(f'Found {len(genres)} genre(s): {genres}')

## 3 · Hyperparameters

In [ ]:
CFG = dict(
    # Data
    seq_len         = 128,
    batch_size      = 128,
    seed            = 42,
    max_poems       = 200,   # cap poems loaded (None = all)
    # Model
    embed_dim       = 64,
    hidden_dim      = 512,
    num_layers      = 2,
    keep_prob       = 0.85,
    # Training
    epochs          = 30,
    steps_per_epoch = 300,
    lr              = 3e-4,
    clip            = 5.0,
    # Evaluation
    val_steps       = 60,
    # Generation
    gen_every       = 5,
    gen_seed        = 'Shall I compare',
    gen_length      = 300,
    temperature     = 0.8,
    top_k           = 40,
)
print(json.dumps(CFG, indent=2))

## 4 · Vocabulary & Data Loader

In [ ]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'


class Vocabulary:
    def __init__(self):
        self.char2idx = {}
        self.idx2char = {}
        self._lookup  = None   # numpy vectorized lookup table

    def build(self, text):
        chars = sorted(set(text))
        all_tokens = [PAD_TOKEN, UNK_TOKEN] + chars
        self.char2idx = {ch: i for i, ch in enumerate(all_tokens)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        # Build an ord->idx array for O(n) vectorized encoding
        max_ord = max(ord(c) for c in self.char2idx if len(c) == 1) + 1
        unk_idx = self.char2idx[UNK_TOKEN]
        lut = np.full(max_ord, unk_idx, dtype=np.int32)
        for ch, idx in self.char2idx.items():
            if len(ch) == 1:
                lut[ord(ch)] = idx
        self._lookup = lut
        self._max_ord = max_ord

    @property
    def size(self):
        return len(self.char2idx)

    def encode_fast(self, text):
        # Vectorized: convert string -> ordinal array -> index array in one shot
        ords = np.frombuffer(text.encode('utf-32-le'), dtype=np.int32)
        unk  = self.char2idx[UNK_TOKEN]
        out  = np.where(ords < self._max_ord, self._lookup[np.minimum(ords, self._max_ord-1)], unk)
        return out.astype(np.int32)

    def encode(self, text):
        unk = self.char2idx[UNK_TOKEN]
        return [self.char2idx.get(ch, unk) for ch in text]

    def decode(self, indices):
        return ''.join(self.idx2char.get(i, UNK_TOKEN) for i in indices)


class DataLoader:
    def __init__(self, data_dir=None, seq_len=128, batch_size=64,
                 seed=42, max_poems=200):
        self.data_dir   = Path(data_dir) if data_dir else GENRES_DIR
        self.seq_len    = seq_len
        self.batch_size = batch_size
        self.max_poems  = max_poems   # cap total poems loaded
        self.rng        = random.Random(seed)
        self.np_rng     = np.random.default_rng(seed)
        self.vocab      = Vocabulary()
        self.splits     = {}
        self.encoded    = {}
        self._load_and_split()
        self._build_vocab()
        self._encode_splits()

    def _load_poems(self):
        poems = []
        txt_files = []
        for genre in os.listdir(self.data_dir):
            genre_path = os.path.join(self.data_dir, genre)
            if not os.path.isdir(genre_path):
                continue
            for f in os.listdir(genre_path):
                if f.endswith('.txt'):
                    txt_files.append(os.path.join(genre_path, f))
        if not txt_files:
            raise FileNotFoundError(f'No .txt files found in {self.data_dir}')
        for path in sorted(txt_files):
            with open(path, encoding='utf-8', errors='replace') as fh:
                raw = fh.read()
            blocks = re.split(r'\n{2,}', raw.strip())
            poems.extend([b.strip() for b in blocks if len(b.strip()) > 50])
            if self.max_poems and len(poems) >= self.max_poems:
                poems = poems[:self.max_poems]
                break
        print(f'  Loaded {len(poems):,} poems from {len(txt_files)} files (cap={self.max_poems})')
        return poems

    def _load_and_split(self):
        poems = self._load_poems()
        self.rng.shuffle(poems)
        n = len(poems)
        n_train = int(0.8 * n)
        n_val   = int(0.1 * n)
        self.splits['train'] = poems[:n_train]
        self.splits['val']   = poems[n_train:n_train + n_val]
        self.splits['test']  = poems[n_train + n_val:]
        for s, v in self.splits.items():
            print(f'  {s}: {len(v):,} poems')

    def _build_vocab(self):
        train_text = '\n'.join(self.splits['train'])
        self.vocab.build(train_text)
        print(f'  Vocabulary size: {self.vocab.size}')

    def _encode_splits(self):
        for split, poems in self.splits.items():
            text = '\n'.join(poems)
            # Fast vectorized encoding — no Python loop over characters
            arr  = self.vocab.encode_fast(text)
            self.encoded[split] = arr
            print(f'  {split} encoded: {len(arr):,} chars')

    def sample_batch(self, split='train'):
        data = self.encoded[split]
        max_start = len(data) - self.seq_len - 1
        if max_start <= 0:
            raise ValueError(f'{split} too short for seq_len={self.seq_len}')
        starts = self.np_rng.integers(0, max_start, size=self.batch_size)
        # Vectorized batch assembly via stride tricks
        idx = starts[:, None] + np.arange(self.seq_len)[None, :]
        X = data[idx]
        Y = data[idx + 1]
        return X, Y


print('Loading data ...')
loader = DataLoader(
    seq_len    = CFG['seq_len'],
    batch_size = CFG['batch_size'],
    seed       = CFG['seed'],
    max_poems  = CFG['max_poems'],
)
vocab  = loader.vocab
V      = vocab.size
print(f'Vocab size V = {V}')

## 5 · GPU-Accelerated Model Components

In [ ]:
# Activation helpers — work on xp arrays (CuPy on GPU, NumPy on CPU)
def sigmoid(x):
    return xp.where(x >= 0, 1.0 / (1.0 + xp.exp(-x)), xp.exp(x) / (1.0 + xp.exp(x)))

def d_sigmoid(s): return s * (1.0 - s)
def d_tanh(t):    return 1.0 - t ** 2


class EmbeddingLayer:
    def __init__(self, vocab_size, embed_dim, seed=0):
        rng   = np.random.default_rng(seed)
        scale = np.sqrt(1.0 / embed_dim)
        self.E  = xp.array(rng.normal(0.0, scale, (vocab_size, embed_dim)))
        self.dE = xp.zeros_like(self.E)
        self._last_tokens = None

    def forward(self, tokens):
        self._last_tokens = tokens
        return self.E[tokens]

    def backward(self, d_out):
        xp.add.at(self.dE, self._last_tokens, d_out)

    def params(self): return {'E': self.E}
    def grads(self):  return {'E': self.dE}
    def zero_grad(self): self.dE[:] = 0.0

    def state_dict(self):
        return {'E': np.array(self.E) if xp is not np else self.E.copy()}

    def load_state_dict(self, d):
        self.E = xp.array(d['E'])


class GRUCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        self.input_dim = input_dim
        self.H         = hidden_dim
        rng = np.random.default_rng(seed)
        D, H = input_dim, hidden_dim

        def xavier(r, c):
            lim = np.sqrt(6.0 / (r + c))
            return xp.array(rng.uniform(-lim, lim, (r, c)))

        self.W_z = xavier(H, H+D);  self.b_z = xp.zeros(H)
        self.W_r = xavier(H, H+D);  self.b_r = xp.zeros(H)
        self.W_h = xavier(H, H+D);  self.b_h = xp.zeros(H)
        self.dW_z = xp.zeros_like(self.W_z); self.db_z = xp.zeros(H)
        self.dW_r = xp.zeros_like(self.W_r); self.db_r = xp.zeros(H)
        self.dW_h = xp.zeros_like(self.W_h); self.db_h = xp.zeros(H)
        self.cache = {}

    def forward(self, x, h_prev):
        xh     = xp.concatenate([h_prev, x])
        z      = sigmoid(self.W_z @ xh + self.b_z)
        r      = sigmoid(self.W_r @ xh + self.b_r)
        rh     = r * h_prev
        xrh    = xp.concatenate([rh, x])
        h_cand = xp.tanh(self.W_h @ xrh + self.b_h)
        h_t    = (1.0 - z) * h_prev + z * h_cand
        self.cache = dict(x=x, h_prev=h_prev, xh=xh, z=z, r=r,
                          rh=rh, xrh=xrh, h_cand=h_cand, h_t=h_t)
        return h_t

    def backward(self, dh_t):
        c = self.cache
        x, h_prev = c['x'], c['h_prev']
        z, r      = c['z'], c['r']
        xrh, h_cand = c['xrh'], c['h_cand']
        H = self.H
        dh_cand  = dh_t * z
        dz       = dh_t * (h_cand - h_prev)
        dh_prev  = dh_t * (1.0 - z)
        d_hc     = dh_cand * d_tanh(h_cand)
        self.dW_h += xp.outer(d_hc, xrh)
        self.db_h += d_hc
        d_xrh    = self.W_h.T @ d_hc
        d_rh     = d_xrh[:H]
        dx_h     = d_xrh[H:]
        dr       = d_rh * h_prev
        dh_prev += d_rh * r
        dz_pre   = dz * d_sigmoid(z)
        self.dW_z += xp.outer(dz_pre, c['xh'])
        self.db_z += dz_pre
        d_xh_z   = self.W_z.T @ dz_pre
        dr_pre   = dr * d_sigmoid(r)
        self.dW_r += xp.outer(dr_pre, c['xh'])
        self.db_r += dr_pre
        d_xh_r   = self.W_r.T @ dr_pre
        d_xh     = d_xh_z + d_xh_r
        dh_prev += d_xh[:H]
        dx       = d_xh[H:] + dx_h
        return dx, dh_prev

    def params(self):
        return dict(W_z=self.W_z, b_z=self.b_z, W_r=self.W_r,
                    b_r=self.b_r, W_h=self.W_h, b_h=self.b_h)

    def grads(self):
        return dict(W_z=self.dW_z, b_z=self.db_z, W_r=self.dW_r,
                    b_r=self.db_r, W_h=self.dW_h, b_h=self.db_h)

    def zero_grad(self):
        for a in (self.dW_z, self.dW_r, self.dW_h, self.db_z, self.db_r, self.db_h):
            a[:] = 0.0

    def state_dict(self):
        return {k: (np.array(v) if xp is not np else v.copy()) for k, v in self.params().items()}

    def load_state_dict(self, d):
        for k in ('W_z','b_z','W_r','b_r','W_h','b_h'):
            setattr(self, k, xp.array(d[k]))


class GRULayer:
    def __init__(self, input_dim, hidden_dim, num_layers=2, keep_prob=0.8, seed=0):
        self.num_layers = num_layers
        self.H          = hidden_dim
        self.keep_prob  = keep_prob
        self.rng        = np.random.default_rng(seed)
        self.cells      = [
            GRUCell(input_dim if i == 0 else hidden_dim, hidden_dim, seed=seed+i)
            for i in range(num_layers)
        ]
        self._masks = [None] * num_layers
        self._h_seq = []

    def forward(self, X, h0=None, training=True):
        T = X.shape[0]
        if h0 is None:
            h0 = xp.zeros((self.num_layers, self.H))
        self._h_seq = []
        layer_input = X
        for li, cell in enumerate(self.cells):
            h_prev = h0[li]
            h_states = []
            for t in range(T):
                h_t = cell.forward(layer_input[t], h_prev)
                h_states.append(h_t)
                h_prev = h_t
            H_layer = xp.stack(h_states)
            self._h_seq.append(H_layer)
            if li < self.num_layers - 1 and training:
                mask = (xp.array(self.rng.random(H_layer.shape)) < self.keep_prob
                        ).astype(xp.float64) / self.keep_prob
                self._masks[li] = mask
                layer_input = H_layer * mask
            else:
                self._masks[li] = None
                layer_input = H_layer
        return layer_input

    def backward(self, dH_out):
        T = dH_out.shape[0]
        dLayer = dH_out
        for li in reversed(range(self.num_layers)):
            cell = self.cells[li]
            mask = self._masks[li]
            if mask is not None:
                dLayer = dLayer * mask
            dh_next = xp.zeros(self.H)
            dInput  = xp.zeros((T, cell.input_dim))
            for t in reversed(range(T)):
                dh_t = dLayer[t] + dh_next
                dx_t, dh_next = cell.backward(dh_t)
                dInput[t] = dx_t
            dLayer = dInput
        return dLayer

    def params(self):
        out = {}
        for i, c in enumerate(self.cells):
            for k, v in c.params().items():
                out[f'layer{i}_{k}'] = v
        return out

    def grads(self):
        out = {}
        for i, c in enumerate(self.cells):
            for k, v in c.grads().items():
                out[f'layer{i}_{k}'] = v
        return out

    def zero_grad(self):
        for c in self.cells: c.zero_grad()

    def state_dict(self):
        out = {}
        for i, c in enumerate(self.cells):
            for k, v in c.state_dict().items():
                out[f'layer{i}_{k}'] = v
        return out

    def load_state_dict(self, d):
        for i, c in enumerate(self.cells):
            cd = {k.replace(f'layer{i}_', ''): v for k, v in d.items() if k.startswith(f'layer{i}_')}
            c.load_state_dict(cd)


class BahdanauAttention:
    def __init__(self, hidden_dim, attn_dim=None, seed=0):
        self.H = hidden_dim
        self.A = attn_dim if attn_dim is not None else hidden_dim // 2
        rng = np.random.default_rng(seed)

        def xavier(r, c):
            lim = np.sqrt(6.0 / (r + c))
            return xp.array(rng.uniform(-lim, lim, (r, c)))

        self.W_a = xavier(self.A, self.H); self.b_a = xp.zeros(self.A)
        self.U_a = xavier(self.A, self.H)
        self.V_a = xavier(self.A, 1)
        self.W_c = xavier(self.H, 2*self.H); self.b_c = xp.zeros(self.H)
        self.dW_a = xp.zeros_like(self.W_a); self.db_a = xp.zeros_like(self.b_a)
        self.dU_a = xp.zeros_like(self.U_a)
        self.dV_a = xp.zeros_like(self.V_a)
        self.dW_c = xp.zeros_like(self.W_c); self.db_c = xp.zeros_like(self.b_c)
        self.cache = {}

    def forward(self, H):
        h_T   = H[-1]
        keys  = H @ self.U_a.T
        query = (self.W_a @ h_T + self.b_a).reshape(1, self.A)
        e_pre = query + keys
        e_tan = xp.tanh(e_pre)
        scores = (e_tan @ self.V_a).squeeze(-1)
        s = scores - scores.max()
        e = xp.exp(s)
        alpha  = e / e.sum()
        c_t    = alpha @ H
        concat = xp.concatenate([h_T, c_t])
        h_attn = xp.tanh(self.W_c @ concat + self.b_c)
        self.cache = dict(H=H, h_T=h_T, keys=keys, query=query,
                          e_pre=e_pre, e_tan=e_tan, scores=scores,
                          alpha=alpha, c_t=c_t, concat=concat, h_attn=h_attn)
        return h_attn, alpha

    def backward(self, d_h_attn):
        c = self.cache
        H, h_T    = c['H'], c['h_T']
        e_tan, alpha = c['e_tan'], c['alpha']
        d_twc = d_h_attn * (1.0 - c['h_attn'] ** 2)
        self.dW_c += xp.outer(d_twc, c['concat'])
        self.db_c += d_twc
        d_concat  = self.W_c.T @ d_twc
        d_hT_cc   = d_concat[:self.H]
        d_c_t     = d_concat[self.H:]
        dH        = xp.outer(alpha, d_c_t)
        d_alpha   = H @ d_c_t
        dot       = alpha @ d_alpha
        d_scores  = alpha * (d_alpha - dot)
        d_e_tan   = xp.outer(d_scores, self.V_a.squeeze())
        self.dV_a += e_tan.T @ d_scores.reshape(-1, 1)
        d_e_pre   = d_e_tan * (1.0 - e_tan ** 2)
        self.dU_a += d_e_pre.T @ H
        dH        += d_e_pre @ self.U_a
        d_query   = d_e_pre.sum(axis=0)
        self.dW_a += xp.outer(d_query, h_T)
        self.db_a += d_query
        dH[-1]    += d_hT_cc + self.W_a.T @ d_query
        return dH

    def params(self):
        return dict(W_a=self.W_a, U_a=self.U_a, V_a=self.V_a, b_a=self.b_a,
                    W_c=self.W_c, b_c=self.b_c)

    def grads(self):
        return dict(W_a=self.dW_a, U_a=self.dU_a, V_a=self.dV_a, b_a=self.db_a,
                    W_c=self.dW_c, b_c=self.db_c)

    def zero_grad(self):
        for a in (self.dW_a, self.dU_a, self.dV_a, self.db_a, self.dW_c, self.db_c):
            a[:] = 0.0

    def state_dict(self):
        return {k: (np.array(v) if xp is not np else v.copy()) for k, v in self.params().items()}

    def load_state_dict(self, d):
        for k in ('W_a','U_a','V_a','b_a','W_c','b_c'):
            setattr(self, k, xp.array(d[k]))


print('Model components defined.')

In [ ]:
def sample_token(logits, temperature=1.0, top_k=0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    if temperature == 0.0:
        idx = int(xp.argmax(logits))
        return idx if xp is np else int(xp.asnumpy(xp.argmax(logits)))
    logits = logits / temperature
    if top_k > 0:
        lg_np = np.array(xp.asnumpy(logits) if xp is not np else logits)
        threshold = np.partition(lg_np, -top_k)[-top_k]
        logits = xp.where(logits >= threshold, logits, xp.array(-np.inf))
    logits = logits - logits.max()
    probs  = xp.exp(logits)
    probs  = probs / probs.sum()
    probs_np = np.array(xp.asnumpy(probs) if xp is not np else probs)
    return int(rng.choice(len(probs_np), p=probs_np))


class Adam:
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, epsilon=1e-8, clip=5.0):
        self.lr = lr; self.beta1 = beta1; self.beta2 = beta2
        self.epsilon = epsilon; self.clip = clip
        self.t = 0; self.m = {}; self.v = {}

    def _clip(self, grads):
        if self.clip is None: return grads
        norm = float(xp.sqrt(sum(xp.sum(g**2) for g in grads.values())))
        if norm > self.clip:
            s = self.clip / (norm + 1e-12)
            return {k: g*s for k, g in grads.items()}
        return grads

    def step(self, params, grads):
        self.t += 1
        grads = self._clip(grads)
        b1t = self.beta1 ** self.t
        b2t = self.beta2 ** self.t
        for key, g in grads.items():
            if key not in params: continue
            if key not in self.m:
                self.m[key] = xp.zeros_like(g)
                self.v[key] = xp.zeros_like(g)
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * g
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * g**2
            m_hat = self.m[key] / (1 - b1t)
            v_hat = self.v[key] / (1 - b2t)
            params[key] -= self.lr * m_hat / (xp.sqrt(v_hat) + self.epsilon)
        return params


print('Optimiser defined.')

In [ ]:
class RNNLM:
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256,
                 num_layers=2, keep_prob=0.8, attn_dim=None, seed=0):
        self.H = hidden_dim
        rng = np.random.default_rng(seed)
        self.embedding = EmbeddingLayer(vocab_size, embed_dim, seed=seed)
        self.gru       = GRULayer(embed_dim, hidden_dim, num_layers=num_layers,
                                  keep_prob=keep_prob, seed=seed)
        self.attention = BahdanauAttention(hidden_dim, attn_dim=attn_dim, seed=seed)
        lim = np.sqrt(6.0 / (hidden_dim + vocab_size))
        self.W_out  = xp.array(rng.uniform(-lim, lim, (vocab_size, hidden_dim)))
        self.b_out  = xp.zeros(vocab_size)
        self.dW_out = xp.zeros_like(self.W_out)
        self.db_out = xp.zeros_like(self.b_out)
        self._cache = {}

    def forward_seq(self, tokens, targets=None, training=True):
        T   = len(tokens)
        emb = self.embedding.forward(tokens)
        H_s = self.gru.forward(emb, training=training)
        h_a, alpha = self.attention.forward(H_s)
        logits_s = self.W_out @ h_a + self.b_out
        logits   = xp.tile(logits_s, (T, 1))
        logits  -= logits.max(axis=1, keepdims=True)
        e        = xp.exp(logits)
        probs    = e / e.sum(axis=1, keepdims=True)
        loss     = 0.0
        if targets is not None:
            log_p = xp.log(probs[xp.arange(T), targets] + 1e-12)
            loss  = float(-log_p.mean())
        self._cache = dict(tokens=tokens, targets=targets, emb=emb,
                           H_states=H_s, h_attn=h_a, alpha=alpha,
                           logits_single=logits_s, probs=probs, T=T)
        return probs, loss

    def backward_seq(self, scale=1.0):
        c = self._cache
        probs, targets, T = c['probs'], c['targets'], c['T']
        d_logits  = probs.copy()
        d_logits[xp.arange(T), targets] -= 1.0
        d_logits  = d_logits / T * scale
        d_ls      = d_logits.sum(axis=0)
        self.dW_out += xp.outer(d_ls, c['h_attn'])
        self.db_out += d_ls
        d_h_attn   = self.W_out.T @ d_ls
        dH_states  = self.attention.backward(d_h_attn)
        d_emb      = self.gru.backward(dH_states)
        self.embedding.backward(d_emb)
        grads = {'W_out': self.dW_out, 'b_out': self.db_out}
        grads.update({f'emb_{k}': v for k, v in self.embedding.grads().items()})
        grads.update({f'gru_{k}': v for k, v in self.gru.grads().items()})
        grads.update({f'attn_{k}': v for k, v in self.attention.grads().items()})
        return grads

    def zero_grad(self):
        self.dW_out[:] = 0.0; self.db_out[:] = 0.0
        self.embedding.zero_grad()
        self.gru.zero_grad()
        self.attention.zero_grad()

    def train_step(self, X_batch, Y_batch):
        """Accumulate gradients across all B sequences before returning."""
        B = X_batch.shape[0]
        self.zero_grad()
        total_loss = 0.0
        scale = 1.0 / B
        for i in range(B):
            _, loss = self.forward_seq(X_batch[i], Y_batch[i], training=True)
            total_loss += loss
            # backward accumulates into self.dW_out etc — call EVERY iteration
            self.backward_seq(scale=scale)
        # Return accumulated grads (they live in model buffers)
        grads = {'W_out': self.dW_out, 'b_out': self.db_out}
        grads.update({f'emb_{k}': v for k, v in self.embedding.grads().items()})
        grads.update({f'gru_{k}': v for k, v in self.gru.grads().items()})
        grads.update({f'attn_{k}': v for k, v in self.attention.grads().items()})
        return total_loss / B, grads

    def generate(self, seed_tokens, vocab, n=200, temperature=1.0, top_k=0, seed=None):
        rng    = np.random.default_rng(seed)
        tokens = list(seed_tokens)
        alpha_last = np.zeros(len(seed_tokens))
        for _ in range(n):
            inp  = xp.array(tokens[-512:], dtype=xp.int32)
            probs, _ = self.forward_seq(inp, targets=None, training=False)
            a = self._cache['alpha']
            alpha_last = a.get() if xp is not np else np.array(a)
            logits = xp.log(probs[-1] + 1e-12)
            tokens.append(sample_token(logits, temperature, top_k, rng))
        return vocab.decode(tokens[len(seed_tokens):]), alpha_last

    def params(self):
        p = {'W_out': self.W_out, 'b_out': self.b_out}
        p.update({f'emb_{k}': v for k, v in self.embedding.params().items()})
        p.update({f'gru_{k}': v for k, v in self.gru.params().items()})
        p.update({f'attn_{k}': v for k, v in self.attention.params().items()})
        return p

    def save(self, path):
        path = str(path)
        if not path.endswith('.npz'): path += '.npz'
        # .get() pulls CuPy array to CPU; no-op for NumPy
        to_save = {k: (v.get() if xp is not np else v) for k, v in self.params().items()}
        np.savez_compressed(path, **to_save)
        print(f'  Saved -> {path}')

    def load(self, path):
        path = str(path)
        if not path.endswith('.npz'): path += '.npz'
        data = np.load(path)
        self.embedding.load_state_dict({k[4:]: v for k, v in data.items() if k.startswith('emb_')})
        self.gru.load_state_dict({k[4:]: v for k, v in data.items() if k.startswith('gru_')})
        self.attention.load_state_dict({k[5:]: v for k, v in data.items() if k.startswith('attn_')})
        self.W_out = xp.array(data['W_out'])
        self.b_out = xp.array(data['b_out'])
        print(f'  Loaded <- {path}')


print('RNNLM defined.')


## 6 · Instantiate Model

In [ ]:
model = RNNLM(
    vocab_size  = V,
    embed_dim   = CFG['embed_dim'],
    hidden_dim  = CFG['hidden_dim'],
    num_layers  = CFG['num_layers'],
    keep_prob   = CFG['keep_prob'],
    seed        = CFG['seed'],
)
optim = Adam(lr=CFG['lr'], clip=CFG['clip'])

# .size works on both CuPy and NumPy arrays — no implicit conversion
total_params = sum(v.size for v in model.params().values())
print(f'Model parameters: {total_params:,}')

RESUME = False
CHECKPOINT_NAME = 'best_model'
if RESUME:
    model.load(str(CHECKPOINTS / CHECKPOINT_NAME))
    print('Resumed from checkpoint.')


## 7 · Training Loop

In [ ]:
history = {'train_loss': [], 'val_ppl': [], 'epoch': []}
best_val_ppl = float('inf')


def estimate_val_ppl(steps=10):
    """Fast val estimate: run `steps` batches, average loss -> perplexity.
    Each batch is ONE forward pass on one sequence — not 128.
    """
    total = 0.0
    for _ in range(steps):
        X_np, Y_np = loader.sample_batch('val')
        # Just use the first sequence of the batch for speed
        x = xp.array(X_np[0], dtype=xp.int32)
        y = xp.array(Y_np[0], dtype=xp.int32)
        _, loss = model.forward_seq(x, y, training=False)
        total += loss
    return float(np.exp(total / steps))


print(f'Starting training for {CFG["epochs"]} epochs ...\n')
t0 = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    epoch_loss = 0.0
    for step in range(CFG['steps_per_epoch']):
        X_np, Y_np = loader.sample_batch('train')
        X = xp.array(X_np, dtype=xp.int32)
        Y = xp.array(Y_np, dtype=xp.int32)
        loss, grads = model.train_step(X, Y)
        optim.step(model.params(), grads)
        epoch_loss += loss
        if (step + 1) % 50 == 0:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch:3d} | Step {step+1:4d}/{CFG["steps_per_epoch"]} '
                  f'| loss {loss:.4f} | {elapsed:.0f}s elapsed', end='\r')

    avg_loss = epoch_loss / CFG['steps_per_epoch']
    val_ppl  = estimate_val_ppl(steps=10)
    history['epoch'].append(epoch)
    history['train_loss'].append(avg_loss)
    history['val_ppl'].append(val_ppl)
    elapsed = time.time() - t0
    print(f'Epoch {epoch:3d}/{CFG["epochs"]} | train_loss {avg_loss:.4f} | val_ppl {val_ppl:.2f} | {elapsed:.0f}s')

    # Save best checkpoint
    if val_ppl < best_val_ppl:
        best_val_ppl = val_ppl
        model.save(str(CHECKPOINTS / 'best_model'))
        print(f'  ** New best val_ppl = {best_val_ppl:.2f} — checkpoint saved **')

    # Save every 5 epochs regardless
    if epoch % 5 == 0:
        model.save(str(CHECKPOINTS / f'epoch_{epoch:03d}'))

    # Generate a sample
    if epoch % CFG['gen_every'] == 0:
        seed_ids = np.array(vocab.encode(CFG['gen_seed']), dtype=np.int32)
        gen_text, _ = model.generate(seed_ids, vocab, n=CFG['gen_length'],
                                     temperature=CFG['temperature'], top_k=CFG['top_k'])
        print(f'\n-- Sample (epoch {epoch}) --')
        print(CFG['gen_seed'] + gen_text)
        print('---\n')

print('\nTraining complete.')


## 8 · Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['epoch'], history['train_loss'], 'b-o', markersize=4)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy')
ax1.grid(True, alpha=0.3)
ax2.plot(history['epoch'], history['val_ppl'], 'r-o', markersize=4)
ax2.set_title('Validation Perplexity'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Perplexity')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'training_curves.png'), dpi=150)
plt.show()
print(f'Best val perplexity: {best_val_ppl:.2f}')

## 9 · Generate Poetry

In [ ]:
model.load(str(CHECKPOINTS / 'best_model'))

PROMPTS = [
    'The moon rises over',
    'In the heart of Africa',
    'When silence falls like snow',
]

for prompt in PROMPTS:
    seed_ids = np.array(vocab.encode(prompt), dtype=np.int32)
    gen_text, alpha = model.generate(
        seed_ids, vocab, n=400,
        temperature=CFG['temperature'], top_k=CFG['top_k'], seed=CFG['seed'])
    print('=' * 60)
    print(f'PROMPT: "{prompt}"')
    print('=' * 60)
    print(prompt + gen_text)
    print()

## 10 · Attention Heatmap

In [ ]:
def plot_attention(prompt, n=80, temperature=0.7, top_k=20):
    seed_ids = np.array(vocab.encode(prompt), dtype=np.int32)
    gen_text, alpha = model.generate(seed_ids, vocab, n=n,
                                     temperature=temperature, top_k=top_k)
    full_seq = list(prompt) + list(gen_text)
    labels   = full_seq[:len(alpha)]
    fig, ax  = plt.subplots(figsize=(max(10, len(labels) * 0.35), 1.8))
    ax.imshow(alpha.reshape(1, -1), aspect='auto', cmap='Blues')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([repr(c)[1:-1] for c in labels], rotation=90, fontsize=7)
    ax.set_yticks([])
    ax.set_title('Bahdanau Attention Weights')
    plt.tight_layout()
    plt.savefig(str(DRIVE_ROOT / 'attention_heatmap.png'), dpi=150)
    plt.show()

plot_attention('Roses are red')